<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/split_video_voice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 — install

!apt-get update -qq
!apt-get install -y -qq ffmpeg

!pip install -q faster-whisper soundfile

In [ ]:
# 2 — setting path

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/jarwo_tts")

VIDEO = ROOT / "source/video.mp4"

DATASET = ROOT / "video_dataset"
WAV_DIR = DATASET / "wavs"

DATASET.mkdir(parents=True, exist_ok=True)
WAV_DIR.mkdir(parents=True, exist_ok=True)

FULL_AUDIO = DATASET / "full_audio.wav"

print("Video :", VIDEO)
print("Output:", DATASET)

In [ ]:
# 3 — video → audio
import subprocess

subprocess.run([
    "ffmpeg",
    "-y",
    "-i", str(VIDEO),
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    "-sample_fmt", "s16",
    str(FULL_AUDIO)
], check=True)

print("Audio extracted:", FULL_AUDIO)

In [ ]:
# 4 — load Whisper Bahasa Indonesia

from faster_whisper import WhisperModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    compute_type = "float16"
else:
    compute_type = "int8"

model = WhisperModel(
    "small",
    device=device,
    compute_type=compute_type
)

print("Device:", device)

In [ ]:
# 5 — transcribe Bahasa Indonesia
segments, info = model.transcribe(
    str(FULL_AUDIO),

    language="id",

    beam_size=5,

    vad_filter=True,

    vad_parameters=dict(
        min_silence_duration_ms=400
    )
)

segments = list(segments)

print("Detected language:", info.language)
print("Segments:", len(segments))

In [ ]:
# Check Hasil
for i, segment in enumerate(segments[:30], 1):
    print(
        i,
        f"{segment.start:.2f} -> {segment.end:.2f}",
        segment.text.strip()
    )

In [ ]:
# 7 — otomatis potong menjadi training WAV
import subprocess
import re
from pathlib import Path

metadata = []

MIN_DURATION = 1.0
MAX_DURATION = 10.0

counter = 1

for segment in segments:

    start = float(segment.start)
    end = float(segment.end)
    duration = end - start

    text = segment.text.strip()

    # bersihkan whitespace
    text = re.sub(r"\s+", " ", text)

    # skip yang terlalu pendek/panjang
    if duration < MIN_DURATION:
        continue

    if duration > MAX_DURATION:
        continue

    # skip text terlalu kecil
    if len(text) < 5:
        continue

    # Piper menggunakan | sebagai delimiter
    if "|" in text:
        text = text.replace("|", " ")

    filename = f"{counter:06d}.wav"
    output = WAV_DIR / filename

    subprocess.run([
        "ffmpeg",
        "-y",
        "-loglevel", "error",

        "-ss", str(start),
        "-to", str(end),

        "-i", str(FULL_AUDIO),

        "-ac", "1",
        "-ar", "22050",
        "-sample_fmt", "s16",

        str(output)
    ], check=True)

    metadata.append(
        f"{filename}|{text}"
    )

    counter += 1

print("Generated:", len(metadata), "clips")

In [ ]:
# 8 Buat Meta data

METADATA = DATASET / "metadata.csv"

with open(
    METADATA,
    "w",
    encoding="utf-8"
) as f:

    for row in metadata:
        f.write(row + "\n")

print(METADATA)

In [ ]:
# 9 Dengarkan Sample Acak

import random
from IPython.display import Audio, display

samples = random.sample(
    list(WAV_DIR.glob("*.wav")),
    min(10, len(list(WAV_DIR.glob("*.wav"))))
)

for wav in samples:

    print(wav.name)

    display(
        Audio(
            str(wav),
            autoplay=False
        )
    )

In [ ]:
# 10 — cek durasi dataset
import soundfile as sf
import numpy as np

durations = []

for wav in WAV_DIR.glob("*.wav"):

    info = sf.info(wav)

    durations.append(
        info.frames / info.samplerate
    )

print("Clips       :", len(durations))
print("Total menit :", round(sum(durations) / 60, 2))
print("Average     :", round(np.mean(durations), 2), "sec")
print("Min         :", round(min(durations), 2))
print("Max         :", round(max(durations), 2))